## Imports

In [1]:
from pathlib import Path
import sys
import os

import numpy as np

import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule

In [2]:
from common import read_file_str, show_formatted_cpp, replace_constants_in_kernel

In [3]:
!cl

usage: cl [ option... ] filename... [ /link linkoption... ]


Microsoft (R) C/C++ Optimizing Compiler Version 19.43.34810 for x64
Copyright (C) Microsoft Corporation.  All rights reserved.



## Parameters

In [4]:
project_working_dir = str(Path(sys.path[0]).parent)
sys.path += [project_working_dir]
os.chdir(project_working_dir)

## Create dummy data for mesh

In [5]:
accelerations = np.array(
    [
        [0, 0, 0],
        [3, 4, 0],
        [-1, -1, -1],
    ],
    dtype=np.float32,
)

In [6]:
velocities = np.array(
    [
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
    ],
    dtype=np.float32,
)

In [7]:
vertices = np.array(
    [
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
    ],
    dtype=np.float32,
)

## Cuda Parameters

In [8]:
BLOCK_SIZE = 1024
NR_BLOCKS = (len(accelerations) + BLOCK_SIZE - 1) // BLOCK_SIZE

## Compile cuda kernel

In [9]:
cuda_code = read_file_str("./profiling/kernels/update_position.cu")

In [10]:
parameter_updates = {"TIME_DELTA": 1, "TERMINAL_VELOCITY": 2}

In [11]:
cuda_code = replace_constants_in_kernel(cuda_code, parameter_updates)

In [12]:
show_formatted_cpp(cuda_code)

In [13]:
mod = SourceModule(cuda_code)

C:\Users\CYBORG\AppData\Local\Temp\ipykernel_9176\3464942708.py:1: UserWarning: The CUDA compiler succeeded, but said the following:
kernel.cu

  mod = SourceModule(cuda_code)


## Set-up memory for running kernel

In [14]:
update_position = mod.get_function("update_position_with_friction")

In [15]:
nr_vertices = np.uint32(len(vertices))
dampening = np.float32(0.5)

### Allocate memory to gpu

In [16]:
assert accelerations.flatten().flags["C_CONTIGUOUS"]
assert velocities.flatten().flags["C_CONTIGUOUS"]
assert vertices.flatten().flags["C_CONTIGUOUS"]

In [17]:
vertices_gpu = cuda.mem_alloc(vertices.nbytes)
velocities_gpu = cuda.mem_alloc(velocities.nbytes)
accelerations_gpu = cuda.mem_alloc(accelerations.nbytes)

In [18]:
cuda.memcpy_htod(vertices_gpu, vertices.flatten())
cuda.memcpy_htod(velocities_gpu, velocities.flatten())
cuda.memcpy_htod(accelerations_gpu, accelerations.flatten())

## Create function

In [19]:
def update_position_kernel():
    update_position(
        accelerations_gpu,
        velocities_gpu,
        vertices_gpu,
        nr_vertices,
        dampening,
        block=(BLOCK_SIZE, 1, 1),
        grid=(NR_BLOCKS, 1, 1),
    )

## Check output is as expected

In [20]:
update_position_kernel()

In [21]:
cuda.memcpy_dtoh(vertices, vertices_gpu)
cuda.memcpy_dtoh(velocities, velocities_gpu)
cuda.memcpy_dtoh(accelerations, accelerations_gpu)

In [22]:
accelerations

array([[ 0.,  0.,  0.],
       [ 3.,  4.,  0.],
       [-1., -1., -1.]], dtype=float32)

In [23]:
velocities

array([[ 0. ,  0. ,  0. ],
       [ 1.2,  1.6,  0. ],
       [-0.5, -0.5, -0.5]], dtype=float32)

Assert velocities, on stationary do not change and max velocity is respected.

In [24]:
assert np.isclose(np.linalg.norm(velocities[0]), 0)

In [25]:
assert np.isclose(np.linalg.norm(velocities[1]), 2.0)

In [26]:
assert np.isclose(np.linalg.norm(velocities[2]), np.sqrt(3) / 2)

Since time step is 1 and starting from origin, velocity and position is the same.

In [27]:
assert np.isclose(vertices, velocities).all()

## Profile function

At this point computation is so small that it seems fixed.

In [28]:
%timeit update_position_kernel()

14 µs ± 317 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
